# **5. Beta Distribution**

## What is the Beta Distribution?
The **Beta distribution** is a continuous probability distribution bounded between 0 and 1. It's commonly used to model:
- Probabilities themselves (meta-probability)
- Proportions and percentages
- Bayesian priors for binomial proportions

## The Formula
$$f(x; \alpha, \beta) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha, \beta)}$$

Where:
- $x \in [0, 1]$ (the probability value)
- $\alpha$ = shape parameter (related to successes)
- $\beta$ = shape parameter (related to failures)
- $B(\alpha, \beta)$ = Beta function (normalization constant)

## Key Parameters
| Parameter | Formula |
|-----------|--------|
| **Mean** | $\mu = \frac{\alpha}{\alpha + \beta}$ |
| **Mode** | $\frac{\alpha - 1}{\alpha + \beta - 2}$ (for $\alpha, \beta > 1$) |
| **Variance** | $\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$ |

## Connection to Binomial
The Beta distribution is the **conjugate prior** for the Binomial:
- **Prior**: Beta($\alpha$, $\beta$)
- **Data**: k successes, n-k failures
- **Posterior**: Beta($\alpha + k$, $\beta + n - k$)

In [ ]:
# ===============================================
# BETA DISTRIBUTION: DIFFERENT SHAPES
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

x = np.linspace(0.001, 0.999, 200)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Different parameter combinations
params = [
    [(1, 1, 'Uniform'), (2, 2, 'Symmetric'), (5, 5, 'Peaked Center')],
    [(2, 5, 'Skewed Left'), (5, 2, 'Skewed Right'), (0.5, 0.5, 'U-shaped')],
    [(1, 3, 'α=1, β=3'), (3, 1, 'α=3, β=1'), (10, 10, 'Very Peaked')],
    [(50, 50, 'Very Concentrated'), (2, 8, 'Strong Left Skew'), (8, 2, 'Strong Right Skew')]
]

for ax, param_set in zip(axes.flatten(), params):
    for alpha, beta, label in param_set:
        y = stats.beta.pdf(x, alpha, beta)
        ax.plot(x, y, linewidth=2, label=f'{label} (α={alpha}, β={beta})')
    ax.set_xlabel('x (probability)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)

plt.suptitle('Beta Distribution: Various Shapes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ===============================================
# BETA DISTRIBUTION: BASIC CALCULATIONS
# ===============================================

from scipy import stats
import numpy as np

print("=" * 50)
print("BETA DISTRIBUTION CALCULATIONS")
print("=" * 50)

# Example: Model uncertainty about a conversion rate
# Prior belief: around 30% with some uncertainty
alpha = 3
beta = 7

print(f"\nBeta({alpha}, {beta}) parameters")

# Mean
mean = alpha / (alpha + beta)
print(f"\nMean = α/(α+β) = {alpha}/({alpha}+{beta}) = {mean:.4f}")

# Using scipy
dist = stats.beta(alpha, beta)
print(f"Scipy mean: {dist.mean():.4f}")
print(f"Scipy std: {dist.std():.4f}")

# Probability calculations
print(f"\nProbability Calculations:")
print(f"P(X < 0.2) = {dist.cdf(0.2):.4f}")
print(f"P(X > 0.4) = {1 - dist.cdf(0.4):.4f}")
print(f"P(0.2 < X < 0.5) = {dist.cdf(0.5) - dist.cdf(0.2):.4f}")

# Percentiles (credible interval)
lower = dist.ppf(0.025)
upper = dist.ppf(0.975)
print(f"\n95% Credible Interval: [{lower:.4f}, {upper:.4f}]")

In [ ]:
# ===============================================
# BAYESIAN UPDATING WITH BETA DISTRIBUTION
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("=" * 50)
print("BAYESIAN UPDATING: Learning from Data")
print("=" * 50)

# Start with a prior belief about a coin
# Uninformative prior: Beta(1, 1) = Uniform
prior_alpha = 1
prior_beta = 1

print(f"\nPrior: Beta({prior_alpha}, {prior_beta}) - No prior knowledge")

# Observe data: 7 heads, 3 tails
heads = 7
tails = 3

print(f"Data: {heads} heads, {tails} tails")

# Posterior (conjugate update)
post_alpha = prior_alpha + heads
post_beta = prior_beta + tails

print(f"\nPosterior: Beta({post_alpha}, {post_beta})")
print(f"Posterior Mean: {post_alpha/(post_alpha + post_beta):.4f}")

# Visualize
x = np.linspace(0, 1, 200)

plt.figure(figsize=(10, 6))

# Prior
prior = stats.beta.pdf(x, prior_alpha, prior_beta)
plt.plot(x, prior, 'b--', linewidth=2, label=f'Prior: Beta({prior_alpha},{prior_beta})')

# Posterior
posterior = stats.beta.pdf(x, post_alpha, post_beta)
plt.plot(x, posterior, 'r-', linewidth=2, label=f'Posterior: Beta({post_alpha},{post_beta})')
plt.fill_between(x, posterior, alpha=0.3, color='red')

# MLE (maximum likelihood)
mle = heads / (heads + tails)
plt.axvline(x=mle, color='green', linestyle=':', linewidth=2, 
            label=f'MLE = {mle:.2f}')

plt.xlabel('Probability of Heads (p)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Bayesian Updating: Prior → Posterior', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.show()

In [ ]:
# ===============================================
# SEQUENTIAL BAYESIAN UPDATING
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Simulate sequential coin flips for a biased coin (true p = 0.6)
np.random.seed(42)
true_p = 0.6
n_flips = 50
flips = np.random.binomial(1, true_p, n_flips)  # 1 = heads, 0 = tails

# Start with uniform prior
alpha = 1
beta = 1

x = np.linspace(0, 1, 200)

fig, ax = plt.subplots(figsize=(12, 6))

# Plot priors at different stages
checkpoints = [0, 5, 10, 20, 50]
colors = plt.cm.viridis(np.linspace(0, 1, len(checkpoints)))

for i, checkpoint in enumerate(checkpoints):
    if checkpoint == 0:
        a, b = 1, 1
    else:
        heads_so_far = sum(flips[:checkpoint])
        tails_so_far = checkpoint - heads_so_far
        a = 1 + heads_so_far
        b = 1 + tails_so_far
    
    y = stats.beta.pdf(x, a, b)
    ax.plot(x, y, linewidth=2, color=colors[i], 
            label=f'After {checkpoint} flips: Beta({a},{b})')

ax.axvline(x=true_p, color='red', linestyle='--', linewidth=2, 
           label=f'True p = {true_p}')

ax.set_xlabel('Probability of Heads (p)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Sequential Bayesian Learning\n(Belief about coin fairness updates with data)', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

print(f"\nFinal data: {sum(flips)} heads, {n_flips - sum(flips)} tails")
print(f"Final posterior mean: {(1 + sum(flips))/(2 + n_flips):.4f}")
print(f"True probability: {true_p}")

In [ ]:
# ===============================================
# APPLICATION: A/B TESTING WITH BETA DISTRIBUTION
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("=" * 50)
print("A/B TESTING: Bayesian Approach")
print("=" * 50)

# Scenario: Testing two button colors
# Version A (blue): 45 conversions, 155 no conversion (n=200)
# Version B (green): 58 conversions, 142 no conversion (n=200)

# Using uniform prior: Beta(1, 1)
alpha_A = 1 + 45    # 46
beta_A = 1 + 155    # 156

alpha_B = 1 + 58    # 59
beta_B = 1 + 142    # 143

print("\nVersion A (Blue Button):")
print(f"  Data: 45/200 conversions")
print(f"  Posterior: Beta({alpha_A}, {beta_A})")
print(f"  Mean: {alpha_A/(alpha_A + beta_A):.1%}")

print("\nVersion B (Green Button):")
print(f"  Data: 58/200 conversions")
print(f"  Posterior: Beta({alpha_B}, {beta_B})")
print(f"  Mean: {alpha_B/(alpha_B + beta_B):.1%}")

# Monte Carlo simulation: P(B > A)
np.random.seed(42)
n_samples = 100000

samples_A = np.random.beta(alpha_A, beta_A, n_samples)
samples_B = np.random.beta(alpha_B, beta_B, n_samples)

prob_B_better = np.mean(samples_B > samples_A)

print(f"\n🎯 P(B > A) = {prob_B_better:.1%}")

# Visualization
x = np.linspace(0.1, 0.5, 200)

plt.figure(figsize=(10, 6))
plt.plot(x, stats.beta.pdf(x, alpha_A, beta_A), 'b-', linewidth=2, 
         label=f'Version A: Beta({alpha_A},{beta_A})')
plt.plot(x, stats.beta.pdf(x, alpha_B, beta_B), 'g-', linewidth=2, 
         label=f'Version B: Beta({alpha_B},{beta_B})')
plt.fill_between(x, stats.beta.pdf(x, alpha_A, beta_A), alpha=0.3, color='blue')
plt.fill_between(x, stats.beta.pdf(x, alpha_B, beta_B), alpha=0.3, color='green')

plt.xlabel('Conversion Rate', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title(f'Bayesian A/B Test\nP(Version B > Version A) = {prob_B_better:.1%}', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

---
## Summary

| Property | Formula/Value |
|----------|---------------|
| **PDF** | $f(x) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha,\beta)}$ |
| **Mean** | $\frac{\alpha}{\alpha + \beta}$ |
| **Bayesian Update** | Prior + Data → Posterior |
| **Conjugate Prior** | Beta(α,β) + Binomial(k,n-k) → Beta(α+k, β+n-k) |
| **SciPy** | `stats.beta(alpha, beta)` |

### Key Insights
- Beta models **uncertainty about probabilities**
- α represents "pseudo-successes", β represents "pseudo-failures"
- As data increases, posterior becomes more concentrated
- Perfect for **Bayesian A/B testing**